# 02 - PennyLane Reference API (Small-Qubit Mode)

Use this when you want an exact/reference result for debugging.

API behavior:
- `qc.expvals_reference(thetas)` -> exact expvals for the circuit built with `Circuit`
- `pennylane_sample(circuit=..., thetas=..., n_qubits=..., shots=...)` -> bitstring samples

Scope: intentionally small-qubit (default <=20) for manageable exact simulation cost.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
from padopauli import Circuit, build_quasi_sampler, pennylane_sample

In [2]:
# Small circuit, built with the Circuit builder API
n_qubits = 4
qc = Circuit(n_qubits)
qc.h(0)
qc.cnot(0, 1)
qc.rzz(1, 2, param_idx=0)
qc.rxx(2, 3, param_idx=1)

observables = [("Z", [q]) for q in range(n_qubits)]

thetas = torch.tensor([0.3, -0.4], dtype=torch.float64)

In [3]:
# Compile tensor expval program via the builder
program = qc.compile(observables=observables, preset='gpu')

tensor_expvals = qc.expvals(thetas)
pl_expvals = qc.expvals_reference(thetas)

print('tensor expvals    :', tensor_expvals)
print('pennylane (exact) :', pl_expvals)
print('max abs diff tensor-vs-pl:', float(torch.max(torch.abs(tensor_expvals.cpu() - pl_expvals.cpu()))))

propagate:   0%|          | 0/4 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 8


zero-filter:   0%|          | 0/4 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 2 (25.000000% of peak)


tensor expvals    : tensor([0.0000, 0.0000, 0.9211, 0.9211], device='cuda:0', dtype=torch.float64)
pennylane (exact) : tensor([0.0000, 0.0000, 0.9211, 0.9211], device='cuda:0', dtype=torch.float64)
max abs diff tensor-vs-pl: 1.2212453270876722e-15


In [4]:
# `build_quasi_sampler` is a low-level functional-API entry point: it takes a raw gate
# list rather than a Circuit, so we hand it `qc.gates` (the circuit the builder recorded).
z_combos = [[i] for i in range(n_qubits)]
sampler = build_quasi_sampler(
    n_qubits=n_qubits,
    circuit=qc.gates,
    z_combos=z_combos,
    max_order=1,
    preset='gpu'
)

samples = pennylane_sample(circuit=qc.gates, thetas=thetas, n_qubits=n_qubits, shots=16, seed=0)
print('samples shape:', tuple(samples.shape))
print(samples)

propagate:   0%|          | 0/4 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 8


zero-filter:   0%|          | 0/4 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 2 (25.000000% of peak)
samples shape: (16, 4)
tensor([[1, 1, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [0, 0, 0, 0],
        [1, 1, 0, 0],
        [0, 0, 0, 0],
        [1, 1, 0, 0],
        [0, 0, 0, 0]], dtype=torch.uint8)
